## 10.17.2026 -- 5:36 PM notes

Right now, I'm using OpenCV in VSCode to be able to run a live server that can track hands.

Must have anaconda installed or else this won't work (obviously) since you need a Python kernel.

If you get the error `ImportError: numpy.core.multiarray failed to import`, this means you need to create a fresh conda environment. 

**Steps to do that:**
- Open Anaconda Prompt (WinKey + "Anaconda Prompt")
- Run these one at a time: 

`conda create -n handtrack python=3.11 -y`
`conda activate handtrack`
`pip install numpy mediapipe opencv-python ipykernel`
`python -m ipykernel install --user --name handtrack`

After `conda activate handtrack`, the line should start with `(handtrack)`

Then, open VSCode, close and reopen VS Code (or press Ctrl+Shift+P and run Developer: Reload Window).
Then, click "Select Kernel," click "Select Another Kernel," and then type in "Jupyter Kernel" (or "Python Environments"), and pick "handtrack"

In [1]:
pip install opencv-python mediapipe numpy

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install pyautogui

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installi

In [2]:
import pyautogui

In [3]:
pyautogui.PAUSE = 0 # for no delay
SCREEN_W, SCREEN_H = pyautogui.size() # get the screen size
MARGIN = 0.15 # the active region is 15% to 85% of the screen
SMOOTH = 0.8 # higher this is, the smoother it is !!!
sx, sy = SCREEN_W / 2, SCREEN_H / 2 # the center of the screen

In [4]:
def to_screen(lm):
    nx = (lm.x - MARGIN) / (1 - 2 * MARGIN)
    ny = (lm.y - MARGIN) / (1 - 2 * MARGIN)
    nx = max(0, min(1, nx))
    ny = max(0, min(1, ny))
    return nx * SCREEN_W, ny * SCREEN_H 

In [6]:
import time
import urllib.request
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# Download the model once (works locally and in Colab)
MODEL_PATH = "hand_landmarker.task"
URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
try:
    open(MODEL_PATH, "rb").close()
except FileNotFoundError:
    urllib.request.urlretrieve(URL, MODEL_PATH)

# Landmark index pairs forming the hand skeleton
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # index
    (5, 9), (9, 10), (10, 11), (11, 12),    # middle
    (9, 13), (13, 14), (14, 15), (15, 16),  # ring
    (13, 17), (17, 18), (18, 19), (19, 20), # pinky
    (0, 17),                                # palm edge
]

def draw_landmarks(frame, hand_landmarks):
    h, w = frame.shape[:2]
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], (0, 255, 0), 2)
    for p in pts:
        cv2.circle(frame, p, 4, (0, 0, 255), -1)

# Create the landmarker (VIDEO mode for webcam frames)
options = vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_tracking_confidence=0.5,
)

cap = cv2.VideoCapture(0)
start = time.time()
WINDOW = "Hand_Detection"
try: 
    with vision.HandLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Wrap the frame and give it a strictly increasing timestamp (ms)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            timestamp_ms = int((time.time() - start) * 1000)

            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            # mapping the landmarks from the hand onto the screen here
            if result.hand_landmarks: 
                tip = result.hand_landmarks[0][8]  # index finger tip
                tx, ty = to_screen(tip)
                sx = sx * SMOOTH + tx * (1 - SMOOTH)
                sy = sy * SMOOTH + ty * (1 - SMOOTH) 
                # gotta stay 1 px inside the corners to avoid triggering the failsafe!
                pyautogui.moveTo(
                    min(max(sx, 1), SCREEN_W - 2),
                    min(max(sy, 1), SCREEN_H - 2)
                )
            
            if result.hand_landmarks:
                for hand_landmarks in result.hand_landmarks:
                    draw_landmarks(frame, hand_landmarks)

            cv2.imshow("Hand Detection", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

finally:
    cap.release()
    cv2.destroyAllWindows()
    for _ in range(5):      # lets Windows actually process the window close
        cv2.waitKey(1)